# Per Patient Cell Type Proportions

In [ ]:
# Autoreload extension for IPython/Jupyter
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('load_ext', 'autoreload')
        ipython.run_line_magic('autoreload', '2')
except (ImportError, NameError):
    pass

import pandas as pd
import numpy as np
import anndata as ad
import s3fs

import scanpy as sc
import matplotlib.pyplot as plt
import openslide
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter, AllDataCenters

sc.set_figure_params(dpi=200, color_map="viridis_r")
sc.settings.verbosity = 0

from cloudpathlib import S3Path
from scipy.sparse import issparse
from joblib import Parallel, delayed
from tqdm import tqdm

from gbmcollaboration.definitions import (
    DATA_PATH_CHUV_FILES,
    DATA_PATH_UKER_FILES,
    DATA_PATH_CR_FILES,
)
from gbmcollaboration import AllDataCenters

%load_ext rpy2.ipython

### Load Clinical Data

In [ ]:
import pandas as pd
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter

# Load processed clinical tables for all centers and combine into one cohort

def load_processed_clinical(dc, center_name):
    d = dc.load_tabular(exclude_idh1_mut=False)
    clin = d["clinical"]
    key = next((k for k in clin if "processed" in str(k).lower() and "clinical" in str(k).lower()), None)
    df = clin[key].copy() if key else list(clin.values())[0].copy()
    df["center"] = center_name
    return df

ch = load_processed_clinical(CHUVDataCenter, "CHUV")
uk = load_processed_clinical(UKERDataCenter, "UKER")
cr = load_processed_clinical(CRDataCenter, "Charite")

clin_all = pd.concat([ch, uk, cr], ignore_index=True, sort=False)
clin_out = clin_all.copy()

print("Total rows:", len(clin_out))
print("Totals by center:\n", clin_out.groupby("center").size())

### Load Single Cell Data

In [ ]:
data_center = AllDataCenters # or "CHUV", "UKER", "CR"
adata_sc = data_center.load_singlecell()

### Create lognorm layer or combined single cell object

In [ ]:
# create a LogNormalize layer
adata.layers["LogNormalize"] = adata.layers["unnormalized_counts"].copy()
sc.pp.normalize_total(
    adata, layer="LogNormalize", target_sum=1e4
)
sc.pp.log1p(adata, layer="LogNormalize")

### Create new UMAP for combined single cell object

In [ ]:
USE_LAYER     = "LogNormalize"
USE_HVGS      = True
HVG_N_TOP     = 3000
N_PCS         = 50
N_NEIGHBORS   = 15
UMAP_MIN_DIST = 0.3
UMAP_SPREAD   = 1.0
RANDOM_STATE  = 13

adata.obsm.pop("X_umap", None)
adata.uns.pop("neighbors", None)
adata.X = adata.layers[USE_LAYER].copy()

if USE_HVGS:
    sc.pp.highly_variable_genes(adata, n_top_genes=HVG_N_TOP, flavor="seurat_v3", subset=True)

sc.tl.pca(adata, n_comps=N_PCS, svd_solver="arpack")
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS)
sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD, random_state=RANDOM_STATE)

### Save new lognorm and UMAP

In [ ]:
SAVE_DIR = "/home/sagemaker-user/gbm-collaboration_2/analysis"
LAYER = "LogNormalize"
UMAP_KEY = "X_umap"

import os, numpy as np, pandas as pd, scipy.sparse as sp

os.makedirs(SAVE_DIR, exist_ok=True)

M = adata.layers[LAYER]
M = M.tocsr() if sp.issparse(M) else sp.csr_matrix(M)
sp.save_npz(f"{SAVE_DIR}/{LAYER}.npz", M)

pd.Series(adata.obs_names, name="obs").to_csv(f"{SAVE_DIR}/obs_names.csv", index=False)
pd.Series(adata.var_names, name="var").to_csv(f"{SAVE_DIR}/var_names.csv", index=False)

U = adata.obsm[UMAP_KEY]
pd.DataFrame(U, index=adata.obs_names, columns=["UMAP1","UMAP2"]).to_csv(f"{SAVE_DIR}/umap_coords.csv"

### Load new lognorm and UMAP from file

In [ ]:
# %%
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp

LOAD_DIR = "/home/sagemaker-user/gbm-collaboration_2/analysis_sc_merged"
LAYER    = "LogNormalize"
UMAP_KEY = "X_umap"

# File checks
npz_path = f"{LOAD_DIR}/{LAYER}.npz"
if not os.path.exists(npz_path):
    raise FileNotFoundError(f"Missing: {npz_path}")

for fn in ("obs_names.csv", "var_names.csv", "umap_coords.csv"):
    p = f"{LOAD_DIR}/{fn}"
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}")

# Load saved artifacts
obs_saved = pd.read_csv(f"{LOAD_DIR}/obs_names.csv")["obs"].astype(str).values
var_saved = pd.read_csv(f"{LOAD_DIR}/var_names.csv")["var"].astype(str).values
M_saved = sp.load_npz(npz_path).tocsr()
U_saved = pd.read_csv(f"{LOAD_DIR}/umap_coords.csv", index_col=0)

# Shape checks
if M_saved.shape != (len(obs_saved), len(var_saved)):
    raise ValueError(f"M_saved shape {M_saved.shape} != ({len(obs_saved)}, {len(var_saved)})")

# Build lookup dicts for fast indexing
obs_saved_idx = {name: i for i, name in enumerate(obs_saved)}
var_saved_idx = {name: i for i, name in enumerate(var_saved)}

# Current object
obs_cur = adata_sc.obs_names.astype(str).values
var_cur = adata_sc.var_names.astype(str).values

# Find overlapping indices
obs_cur_keep = []
obs_sav_keep = []
for i, name in enumerate(obs_cur):
    if name in obs_saved_idx:
        obs_cur_keep.append(i)
        obs_sav_keep.append(obs_saved_idx[name])

var_cur_keep = []
var_sav_keep = []
for i, name in enumerate(var_cur):
    if name in var_saved_idx:
        var_cur_keep.append(i)
        var_sav_keep.append(var_saved_idx[name])

obs_cur_keep = np.array(obs_cur_keep)
obs_sav_keep = np.array(obs_sav_keep)
var_cur_keep = np.array(var_cur_keep)
var_sav_keep = np.array(var_sav_keep)

if len(obs_cur_keep) == 0:
    raise ValueError("No overlapping obs_names")
if len(var_cur_keep) == 0:
    raise ValueError("No overlapping var_names")

# Extract submatrix and rebuild
sub = M_saved[obs_sav_keep, :][:, var_sav_keep].tocoo()
M_cur = sp.csr_matrix(
    (sub.data, (obs_cur_keep[sub.row], var_cur_keep[sub.col])),
    shape=(len(obs_cur), len(var_cur))
)
adata_sc.layers[LAYER] = M_cur

# Align UMAP
U_cur = np.full((len(obs_cur), 2), np.nan, dtype=float)
U_saved.index = U_saved.index.astype(str)
obs_in_umap = U_saved.index.isin(obs_cur)
U_sub = U_saved.loc[obs_in_umap].iloc[:, :2]
umap_idx = {name: i for i, name in enumerate(obs_cur)}
for name, row in U_sub.iterrows():
    if name in umap_idx:
        U_cur[umap_idx[name], :] = row.values[:2]
adata_sc.obsm[UMAP_KEY] = U_cur

print("Loaded into adata_sc:")
print(f"  layer: {LAYER} {adata_sc.layers[LAYER].shape} nnz: {adata_sc.layers[LAYER].nnz}")
print(f"  umap: {UMAP_KEY} {adata_sc.obsm[UMAP_KEY].shape} nan_rows: {np.isnan(adata_sc.obsm[UMAP_KEY]).any(axis=1).sum()}")
print(f"  obs_overlap: {len(obs_cur_keep)} / {len(obs_cur)}")
print(f"  var_overlap: {len(var_cur_keep)} / {len(var_cur)}")

### Load curated cell type labels from file

In [ ]:
import os
import pandas as pd

# user vars
ADATA_OBJ = adata_sc  # set to adata_sc (single cell) or adata (visium), explicitly
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"

LEVEL4_KEY = "celltype_level4_scanvi"
NEW_COLS = [
    "celltype_level4_curated",
    "celltype_level3_curated",
    "celltype_level3_figure",
    "celltype_level2_curated",
    "celltype_level2_figure",
    "celltype_level1_curated",
]

if not os.path.exists(KEY_PATH):
    raise FileNotFoundError(f"Missing KEY_PATH: {KEY_PATH}")

key = pd.read_csv(KEY_PATH)

# checks
if LEVEL4_KEY not in ADATA_OBJ.obs.columns:
    raise KeyError(f"Missing {LEVEL4_KEY} in ADATA_OBJ.obs")
missing_cols = [c for c in [LEVEL4_KEY] + NEW_COLS if c not in key.columns]
if missing_cols:
    raise KeyError(f"Missing column(s) in key file: {missing_cols}")

# normalize + deduplicate key
key = key.copy()
key[LEVEL4_KEY] = key[LEVEL4_KEY].astype(str).str.strip()
for c in NEW_COLS:
    key[c] = key[c].astype(str).where(key[c].notna(), None)

key = key.drop_duplicates(subset=[LEVEL4_KEY], keep="first").copy()

# mapping dicts from level4 -> curated levels
key_map = key.set_index(LEVEL4_KEY)[NEW_COLS].to_dict(orient="series")

# apply mappings to create new obs columns
src = ADATA_OBJ.obs[LEVEL4_KEY].astype(str).str.strip()

for col in NEW_COLS:
    mapped = src.map(key_map[col])
    out = mapped.fillna("Unmapped").astype(str)

    cats = pd.Index(key[col].dropna().astype(str).unique())
    if "Unmapped" in out.values and "Unmapped" not in cats:
        cats = cats.append(pd.Index(["Unmapped"]))

    ADATA_OBJ.obs[col] = pd.Categorical(out, categories=cats)

# quick report
lvl4_all = pd.Index(src.unique())
lvl4_in_key = pd.Index(key[LEVEL4_KEY].astype(str).unique())
missing = lvl4_all.difference(lvl4_in_key)

print(f"Target object: {'adata_sc' if ADATA_OBJ is adata_sc else 'adata'}")
print(f"Added columns: {NEW_COLS}")
print(f"Level4 labels in target: {len(lvl4_all)}")
print(f"Level4 labels covered by key: {len(lvl4_all.intersection(lvl4_in_key))}")
print(f"Level4 labels missing from key: {len(missing)}")
if len(missing):
    print("Unmapped labels:", ", ".join(list(missing)[:20]) + (" ..." if len(missing) > 20 else ""))

### Compare Clinical and Single Cell data

In [ ]:
import re
import pandas as pd

# Compare clinical vs single cell samples by root ID

MOS_COL = "Tumour block ID for MOSAIC"

def root_id(s):
    s = str(s).strip()
    return re.sub(r'_(vis|vbu|vb2|bu2)$', '', s, flags=re.IGNORECASE)

clin_roots = (
    clin_out[MOS_COL]
    .dropna()
    .astype(str)
    .str.strip()
    .map(root_id)
    .unique()
)

sc_roots = (
    adata.obs["orig.ident"]
    .astype(str)
    .str.strip()
    .map(root_id)
    .unique()
)

set_clin = set(clin_roots)
set_sc = set(sc_roots)

overlap = set_clin & set_sc
only_clin = sorted(set_clin - set_sc)

print(f"Clinical samples (roots): {len(set_clin)}")
print(f"Single cell samples (roots): {len(set_sc)}")
print(f"Overlapping samples: {len(overlap)}")
print(f"Clinical only (not in single cell): {len(only_clin)}")
print("Clinical only list:", ", ".join(only_clin))

### Quantify cell counts per patient

In [ ]:
import re
import pandas as pd

LOAD_DIR = "/home/sagemaker-user/gbm-collaboration_2/analysis"

# Build per-patient cell-type proportion tables for L1, L2, L3, and L4 and save to CSV

def root_id(s):
    s = str(s).strip()
    return re.sub(r'_(vis|vbu|vb2|bu2)$', '', s, flags=re.IGNORECASE)

adata.obs["sample_root"] = adata.obs["orig.ident"].astype(str).map(root_id)

def proportion_table(col):
    assert col in adata.obs.columns, f"Missing {col} in adata.obs"
    counts = adata.obs.groupby(["sample_root", col]).size().unstack(fill_value=0)
    return counts.div(counts.sum(axis=1), axis=0)

# proportions
df_l1 = proportion_table("celltype_level1_curated")
df_l2 = proportion_table("celltype_level2_curated")
df_l3 = proportion_table("celltype_level3_curated")
df_l4 = proportion_table("celltype_level4_curated")

# total cells per sample_root
cells_per_sample = (
    adata.obs.groupby("sample_root")
    .size()
    .rename("n_cells_total")
)

# save
df_l1.to_csv(f"{LOAD_DIR}/sc_celltype_level1_proportions.csv")
df_l2.to_csv(f"{LOAD_DIR}/sc_celltype_level2_proportions.csv")
df_l3.to_csv(f"{LOAD_DIR}/sc_celltype_level3_proportions.csv")
df_l4.to_csv(f"{LOAD_DIR}/sc_celltype_level4_proportions.csv")
cells_per_sample.to_csv(f"{LOAD_DIR}/sc_cells_per_sample.csv")

### Calculate Cox hazard ratios for each cell type proportion

In [ ]:
import re
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

MOS_COL = "Tumour block ID for MOSAIC"

# helper to strip Visium suffixes
def root_id(s):
    s = str(s).strip()
    return re.sub(r'_(vis|vbu|vb2|bu2)$', '', s, flags=re.IGNORECASE)

# clinical table with one row per sample_root
clin_surv = clin_out.copy()
clin_surv["sample_root"] = clin_surv[MOS_COL].astype(str).map(root_id)
clin_surv = clin_surv.dropna(subset=["sample_root", "os_years", "os_event"])
clin_surv = clin_surv.drop_duplicates(subset=["sample_root"])

# if not already created earlier
# df_l1 = proportion_table("celltype_level1_curated")
# df_l2 = proportion_table("celltype_level2_curated")
# df_l3 = proportion_table("celltype_level3_curated")
# df_l4 = proportion_table("celltype_level4_curated")

def cox_table(df_props, label, min_patients=10, min_nonzero=5, min_var=1e-4):
    merged = clin_surv[["sample_root", "os_years", "os_event"]].merge(
        df_props,
        left_on="sample_root",
        right_index=True,
        how="inner",
    )

    results = []
    for col in df_props.columns:
        tmp = merged[["os_years", "os_event", col]].dropna()

        if len(tmp) < min_patients:
            continue
        if (tmp[col] > 0).sum() < min_nonzero:
            continue
        if tmp[col].var() < min_var:
            continue

        try:
            cph = CoxPHFitter()
            cph.fit(tmp, duration_col="os_years", event_col="os_event")
            s = cph.summary.loc[col]

            results.append({
                "cell_type": col,
                "n_patients": int(len(tmp)),
                "n_events": int(tmp["os_event"].sum()),
                "coef": float(s["coef"]),
                "HR": float(s["exp(coef)"]),
                "HR_lower95": float(s["exp(coef) lower 95%"]),
                "HR_upper95": float(s["exp(coef) upper 95%"]),
                "z": float(s["z"]),
                "p": float(s["p"]),
            })
        except Exception as e:
            print(f"Skipping {label} {col} due to fitting error: {e}")

    out = pd.DataFrame(results).sort_values("p")
    out.to_csv(f"{LOAD_DIR}/cox_univariate_{label}.csv", index=False)
    print(f"{label}: {len(out)} cell types tested")
    return out

cox_l1 = cox_table(df_l1, "level1")
cox_l2 = cox_table(df_l2, "level2")
cox_l3 = cox_table(df_l3, "level3")
cox_l4 = cox_table(df_l4, "level4")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

# merge survival and level 2 proportions
surv = clin_surv[["sample_root", "os_years", "os_event"]].dropna()
props = df_l2[["Microglia", "T_NK"]].copy()
km_data = surv.merge(props, left_on="sample_root", right_index=True, how="inner")

def plot_km_by_quantiles(data, col, q_low=0.25, q_high=0.75, ax=None):
    d = data[["os_years", "os_event", col]].dropna()
    q1 = d[col].quantile(q_low)
    q3 = d[col].quantile(q_high)

    low = d[d[col] <= q1]
    high = d[d[col] >= q3]

    print(f"{col}: low n={len(low)}, high n={len(high)}")

    if ax is None:
        fig, ax = plt.subplots()

    kmf = KaplanMeierFitter()
    for group_df, label in [
        (low,  f"{col} low (≤ Q{int(q_low*100)})"),
        (high, f"{col} high (≥ Q{int(q_high*100)})"),
    ]:
        if len(group_df) == 0:
            continue
        kmf.fit(durations=group_df["os_years"],
                event_observed=group_df["os_event"],
                label=label)
        kmf.plot(ax=ax)

    ax.set_xlabel("OS (years)")
    ax.set_ylabel("Survival probability")
    ax.set_title(f"KM by {col} proportion (Q{int(q_low*100)} vs Q{int(q_high*100)})")
    return ax

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

plot_km_by_quantiles(km_data, "Microglia", ax=axes[0])
plot_km_by_quantiles(km_data, "T_NK", ax=axes[1])

plt.tight_layout()
plt.show()

### Cell type correlation matrix

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def clean_corr(corr):
    corr = corr.replace([np.inf, -np.inf], np.nan)
    corr = corr.dropna(axis=0, how="all").dropna(axis=1, how="all")
    common = corr.index.intersection(corr.columns)
    corr = corr.loc[common, common]
    return corr.fillna(0.0)

# drop Doublet columns
l3_cols = [c for c in df_l3.columns if "doublet" not in c.lower()]
l4_cols = [c for c in df_l4.columns if "doublet" not in c.lower()]

df_l3_nd = df_l3[l3_cols].copy()
df_l4_nd = df_l4[l4_cols].copy()

# ------------ level 3 ------------
corr_l3 = df_l3_nd.corr(method="pearson")
corr_l3_clean = clean_corr(corr_l3)
corr_l3_clean.to_csv(f"{LOAD_DIR}/sc_celltype_level3_correlations_noDoublet.csv")

g3 = sns.clustermap(
    corr_l3_clean,
    cmap="seismic",      # or your preferred cmap
    vmin=-1,
    vmax=1,
    linewidths=0,
    figsize=(9, 7),
    cbar=False,
    cbar_pos=None,
    dendrogram_ratio=(0.08, 0.08),
)

ax3 = g3.ax_heatmap
rows3 = g3.data2d.index
cols3 = g3.data2d.columns

ax3.set_yticks(np.arange(len(rows3)))
ax3.set_yticklabels(rows3, rotation=0)

ax3.set_xticks(np.arange(len(cols3)))
ax3.set_xticklabels(cols3, rotation=90)

ax3.tick_params(axis="both", labelsize=7)
g3.fig.tight_layout()
plt.show()

# ------------ level 4 ------------
corr_l4 = df_l4_nd.corr(method="pearson")
corr_l4_clean = clean_corr(corr_l4)
corr_l4_clean.to_csv(f"{LOAD_DIR}/sc_celltype_level4_correlations_noDoublet.csv")

g4 = sns.clustermap(
    corr_l4_clean,
    cmap="seismic",
    vmin=-1,
    vmax=1,
    linewidths=0,
    figsize=(10, 8),
    cbar=False,
    cbar_pos=None,
    dendrogram_ratio=(0.08, 0.08),
)

ax4 = g4.ax_heatmap
rows4 = g4.data2d.index
cols4 = g4.data2d.columns

ax4.set_yticks(np.arange(len(rows4)))
ax4.set_yticklabels(rows4, rotation=0)

ax4.set_xticks(np.arange(len(cols4)))
ax4.set_xticklabels(cols4, rotation=90)

ax4.tick_params(axis="both", labelsize=6)
g4.fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# Build level 4 matrix (patients x cell types), drop Doublets and zero-variance columns
cols_l4 = [c for c in df_l4.columns if "doublet" not in c.lower()]
X4 = df_l4[cols_l4].copy()

var4 = X4.var(axis=0)
cols_keep4 = list(var4[var4 > 0].index)
X4 = X4[cols_keep4]

# Correlation-based clustering of level 4 cell types
corr_l4 = X4.corr().fillna(0.0)

dist4 = 1.0 - corr_l4.values
np.fill_diagonal(dist4, 0.0)
dist4_condensed = squareform(dist4, checks=False)

Z4 = linkage(dist4_condensed, method="average")

# Number of modules (tune this as needed)
N_MODULES_L4 = 8
module_labels4 = fcluster(Z4, N_MODULES_L4, criterion="maxclust")

module_l4 = pd.DataFrame(
    {"cell_type": corr_l4.columns, "module": module_labels4}
).sort_values(["module", "cell_type"])

module_l4.to_csv(f"{LOAD_DIR}/sc_level4_celltype_modules.csv", index=False)
print("Level 4 modules defined:", module_l4["module"].nunique())
print("Module sizes:\n", module_l4["module"].value_counts().sort_index())

# Per-patient module abundance scores (sum of member proportions per module)
df_modules_l4 = pd.DataFrame(index=X4.index)

for m in sorted(module_l4["module"].unique()):
    members = module_l4.loc[module_l4["module"] == m, "cell_type"].tolist()
    df_modules_l4[f"ModuleL4_{m}"] = X4[members].sum(axis=1)

df_modules_l4.to_csv(f"{LOAD_DIR}/sc_level4_module_abundances.csv")

# Merge with survival data
df_surv_mod_l4 = clin_surv[["sample_root", "os_years", "os_event"]].merge(
    df_modules_l4,
    left_on="sample_root",
    right_index=True,
    how="inner",
).dropna(subset=["os_years", "os_event"])

# Univariate Cox for each level 4 module
results_l4 = []
for col in df_modules_l4.columns:
    tmp = df_surv_mod_l4[["os_years", "os_event", col]].dropna()
    if tmp[col].var() == 0 or len(tmp) < 10:
        continue

    cph = CoxPHFitter()
    cph.fit(tmp, duration_col="os_years", event_col="os_event")
    s = cph.summary.loc[col]

    results_l4.append({
        "module": col,
        "n_patients": int(len(tmp)),
        "n_events": int(tmp["os_event"].sum()),
        "coef": float(s["coef"]),
        "HR": float(s["exp(coef)"]),
        "HR_lower95": float(s["exp(coef) lower 95%"]),
        "HR_upper95": float(s["exp(coef) upper 95%"]),
        "z": float(s["z"]),
        "p": float(s["p"]),
    })

cox_modules_l4 = pd.DataFrame(results_l4).sort_values("p")
cox_modules_l4.to_csv(f"{LOAD_DIR}/cox_univariate_level4_modules_OS.csv", index=False)

print("Level 4 modules, OS: tested", len(cox_modules_l4), "modules")
print(cox_modules_l4)

In [ ]:
# Print full composition and mean abundance for each level 4 module

for m in sorted(module_l4["module"].unique()):
    members = module_l4.loc[module_l4["module"] == m, "cell_type"].tolist()
    mean_val = float(module_means_l4.get(f"ModuleL4_{m}", np.nan))
    print(f"Module {m}  (n_celltypes={len(members)}, mean_abundance={mean_val:.4f})")
    for ct in members:
        print(f"  - {ct}")
    print()

In [ ]:
# Mean abundance per module across patients
module_means_l4 = df_modules_l4.mean().sort_values()
print(module_means_l4)

In [ ]:
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

# Modules to use for Cox (multi-cell, reasonably abundant)
modules_for_cox = [1, 2, 3, 4]

cols_for_cox = [f"ModuleL4_{m}" for m in modules_for_cox if f"ModuleL4_{m}" in df_modules_l4.columns]
df_modules_l4_sel = df_modules_l4[cols_for_cox].copy()

# Merge with survival data
df_surv_mod_l4_sel = clin_surv[["sample_root", "os_years", "os_event"]].merge(
    df_modules_l4_sel,
    left_on="sample_root",
    right_index=True,
    how="inner",
).dropna(subset=["os_years", "os_event"])

results_l4_sel = []

for col in df_modules_l4_sel.columns:
    tmp = df_surv_mod_l4_sel[["os_years", "os_event", col]].dropna()
    if tmp[col].var() == 0 or len(tmp) < 10:
        continue

    cph = CoxPHFitter()
    cph.fit(tmp, duration_col="os_years", event_col="os_event")
    s = cph.summary.loc[col]

    results_l4_sel.append({
        "module": col,
        "n_patients": int(len(tmp)),
        "n_events": int(tmp["os_event"].sum()),
        "coef": float(s["coef"]),
        "HR": float(s["exp(coef)"]),
        "HR_lower95": float(s["exp(coef) lower 95%"]),
        "HR_upper95": float(s["exp(coef) upper 95%"]),
        "z": float(s["z"]),
        "p": float(s["p"]),
    })

cox_modules_l4_sel = pd.DataFrame(results_l4_sel).sort_values("p")
cox_modules_l4_sel.to_csv(f"{LOAD_DIR}/cox_univariate_level4_modules_OS_main.csv", index=False)

print("Level 4 multi-module OS Cox results (modules 1–4):")
print(cox_modules_l4_sel)

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster

# df_l4 is the level 4 per-patient cell-type proportion table

# drop doublet cell types
cols_l4 = [c for c in df_l4.columns if not c.startswith("Doublet_")]
df_l4_nd = df_l4[cols_l4].copy()

# drop zero-variance cell types
df_l4_nd = df_l4_nd.loc[:, df_l4_nd.var(axis=0) > 0]

# correlation and clean-up
corr_l4 = df_l4_nd.corr()
corr_l4 = corr_l4.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# distance matrix, enforce symmetry and zero diagonal
dist_mat = 1.0 - corr_l4.values
dist_mat = (dist_mat + dist_mat.T) / 2.0
np.fill_diagonal(dist_mat, 0.0)

# hierarchical clustering into 8 modules (as before)
condensed = squareform(dist_mat, checks=False)
Z_l4 = linkage(condensed, method="average")
labels_l4 = fcluster(Z_l4, t=8, criterion="maxclust")

mod_assign_l4 = pd.Series(labels_l4, index=df_l4_nd.columns, name="module")

print("Level 4 modules defined:", mod_assign_l4.nunique())
print("Module sizes:\n", mod_assign_l4.value_counts().sort_index())

# per-patient module scores: sum of member cell-type proportions
mod_scores_l4 = pd.DataFrame(index=df_l4_nd.index)
for m in sorted(mod_assign_l4.unique()):
    members = mod_assign_l4[mod_assign_l4 == m].index
    mod_scores_l4[f"ModuleL4_{m}"] = df_l4_nd[members].sum(axis=1)

print("Module means:\n", mod_scores_l4.mean().sort_values())

In [ ]:
mod_assign_l4[mod_assign_l4 == 4]

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster

# df_l4_nd: level 4 per-patient proportions (non-doublets, no zero-variance)
# mod_assign_l4: Series mapping each level 4 cell type -> module ID (1..8)

# subset to module 4 members
mod4_members = mod_assign_l4[mod_assign_l4 == 4].index
df_l4_m4 = df_l4_nd[mod4_members].copy()

# drop zero-variance cell types inside module 4
df_l4_m4 = df_l4_m4.loc[:, df_l4_m4.var(axis=0) > 0]

# correlation and distance inside module 4
corr_m4 = df_l4_m4.corr()
corr_m4 = corr_m4.replace([np.inf, -np.inf], np.nan).fillna(0.0)

dist_m4 = 1.0 - corr_m4.values
dist_m4 = (dist_m4 + dist_m4.T) / 2.0
np.fill_diagonal(dist_m4, 0.0)

# cluster into submodules
N_SUB = 4  # try 3 or 4
Z_m4 = linkage(squareform(dist_m4, checks=False), method="average")
sub_labels = fcluster(Z_m4, t=N_SUB, criterion="maxclust")

mod4_sub = pd.Series(sub_labels, index=df_l4_m4.columns, name="submodule")

print("Module 4 submodules:", mod4_sub.nunique())
print("Submodule sizes:\n", mod4_sub.value_counts().sort_index())

for k in sorted(mod4_sub.unique()):
    members = mod4_sub[mod4_sub == k].index.tolist()
    print(f"\nSubmodule 4_{k} members ({len(members)}):")
    print(", ".join(members))

# per-patient submodule scores (sum of member proportions)
mod4_scores = pd.DataFrame(index=df_l4_m4.index)
for k in sorted(mod4_sub.unique()):
    members = mod4_sub[mod4_sub == k].index
    mod4_scores[f"ModuleL4_4_sub{k}"] = df_l4_m4[members].sum(axis=1)

print("\nModule 4 submodule score means:")
print(mod4_scores.mean().sort_values())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

def km_for_module(module_col, q=0.75):
    """KM plot for one module, High = top quantile vs Low = rest."""
    df = (
        clin_surv[["sample_root", "os_years", "os_event"]]
        .merge(
            mod4_scores[[module_col]],
            left_on="sample_root",
            right_index=True,
            how="inner",
        )
        .dropna(subset=["os_years", "os_event", module_col])
    )

    df["os_years"] = pd.to_numeric(df["os_years"], errors="coerce")
    df["os_event"] = pd.to_numeric(df["os_event"], errors="coerce").astype(int)
    df = df.dropna(subset=["os_years", "os_event", module_col])

    scores = df[module_col]
    thr = scores.quantile(q)

    df["group"] = np.where(scores >= thr, "High", "Low")
    gH = df[df["group"] == "High"]
    gL = df[df["group"] == "Low"]

    if len(gH) < 5 or len(gL) < 5:
        print(f"{module_col}: skipped (High={len(gH)}, Low={len(gL)})")
        return

    kmH = KaplanMeierFitter(label=f"{module_col} High")
    kmL = KaplanMeierFitter(label=f"{module_col} Low")

    plt.figure(figsize=(6, 5))
    kmH.fit(gH["os_years"], event_observed=gH["os_event"]).plot(ci_show=True)
    kmL.fit(gL["os_years"], event_observed=gL["os_event"]).plot(ci_show=True)

    lr = logrank_test(
        gH["os_years"], gL["os_years"],
        event_observed_A=gH["os_event"],
        event_observed_B=gL["os_event"],
    )
    p = float(lr.p_value)

    plt.title(f"OS KM: {module_col} High vs Low\nlog-rank p={p:.3g}")
    plt.xlabel("OS (years)")
    plt.ylabel("Survival probability")
    plt.grid(False)
    plt.legend(frameon=False, loc="best")
    plt.tight_layout()
    plt.show()

    print(f"{module_col}: thr(q={q}): {thr:.4g}, High n={len(gH)}, Low n={len(gL)}, p={p:.6g}")
    print(f"  median High={gH[module_col].median():.4g}, Low={gL[module_col].median():.4g}")
    print()

# run for all Module 4 submodules
for col in sorted([c for c in mod4_scores.columns if c.startswith("ModuleL4_4_sub")]):
    km_for_module(col, q=0.75)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

MODULE_COL = "ModuleL4_4_sub2"
MIN_GROUP = 10  # minimum n per arm

# 1) Prepare merged DF
df_mod2 = (
    clin_surv[["sample_root", "os_years", "os_event"]]
    .merge(
        mod4_scores[[MODULE_COL]],
        left_on="sample_root",
        right_index=True,
        how="inner",
    )
    .dropna(subset=["os_years", "os_event", MODULE_COL])
)

df_mod2["os_years"] = pd.to_numeric(df_mod2["os_years"], errors="coerce")
df_mod2["os_event"] = pd.to_numeric(df_mod2["os_event"], errors="coerce").astype(int)
df_mod2 = df_mod2.dropna(subset=["os_years", "os_event", MODULE_COL])

scores = df_mod2[MODULE_COL]

def make_split(df, scheme):
    df = df.copy()

    if scheme == "median":
        thr = scores.median()
        df["group"] = np.where(scores > thr, "High", "Low")
        info = {"threshold": thr}

    elif scheme == "60_40":
        thr = scores.quantile(0.6)
        df["group"] = np.where(scores >= thr, "High", "Low")
        info = {"threshold": thr}

    elif scheme == "70_30":
        thr = scores.quantile(0.7)
        df["group"] = np.where(scores >= thr, "High", "Low")
        info = {"threshold": thr}

    elif scheme == "80_20":
        thr = scores.quantile(0.8)
        df["group"] = np.where(scores >= thr, "High", "Low")
        info = {"threshold": thr}

    elif scheme == "extreme_quartiles":
        q1 = scores.quantile(0.25)
        q3 = scores.quantile(0.75)
        df["group"] = np.where(
            scores <= q1, "Low",
            np.where(scores >= q3, "High", "Mid")
        )
        df = df[df["group"].isin(["High", "Low"])]
        info = {"q1": q1, "q3": q3}

    else:
        raise ValueError("Unknown scheme")

    gH = df[df["group"] == "High"]
    gL = df[df["group"] == "Low"]

    return df, gH, gL, info

schemes = ["median", "60_40", "70_30", "80_20", "extreme_quartiles"]
results = []

print("ModuleL4_4_sub2 split schemes")
for scheme in schemes:
    df_s, gH, gL, info = make_split(df_mod2, scheme)
    nH, nL = len(gH), len(gL)

    if nH < MIN_GROUP or nL < MIN_GROUP:
        p = np.nan
        print(f"  {scheme:16s}: skipped (High={nH}, Low={nL})")
    else:
        lr = logrank_test(
            gH["os_years"], gL["os_years"],
            event_observed_A=gH["os_event"],
            event_observed_B=gL["os_event"],
        )
        p = float(lr.p_value)
        print(
            f"  {scheme:16s}: p={p:.4g}, High n={nH}, Low n={nL}, "
            f"median High={gH[MODULE_COL].median():.4g}, Low={gL[MODULE_COL].median():.4g}"
        )

    results.append((scheme, p, nH, nL))

# pick best scheme by p-value among those that actually ran
valid = [(s, p, nH, nL) for (s, p, nH, nL) in results if np.isfinite(p)]
if not valid:
    raise RuntimeError("No valid split produced enough patients per group.")

best_scheme, best_p, best_nH, best_nL = sorted(valid, key=lambda x: x[1])[0]
print(f"\nBest scheme for {MODULE_COL}: {best_scheme} (p={best_p:.4g}, High n={best_nH}, Low n={best_nL})")

# Rebuild best split and plot KM
df_best, gH_best, gL_best, info_best = make_split(df_mod2, best_scheme)

kmH = KaplanMeierFitter(label=f"{MODULE_COL} High ({best_scheme})")
kmL = KaplanMeierFitter(label=f"{MODULE_COL} Low ({best_scheme})")

plt.figure(figsize=(6, 5))
kmH.fit(gH_best["os_years"], event_observed=gH_best["os_event"]).plot(ci_show=True)
kmL.fit(gL_best["os_years"], event_observed=gL_best["os_event"]).plot(ci_show=True)

plt.title(f"OS KM: {MODULE_COL} High vs Low\n{best_scheme}, log-rank p={best_p:.3g}")
plt.xlabel("OS (years)")
plt.ylabel("Survival probability")
plt.grid(False)
plt.legend(frameon=False, loc="best")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# ------------------------------------------------------------------
# 1) Recreate Module 4 submodule membership from your printed lists
# ------------------------------------------------------------------

sub4_1 = [
    "Macrophage_cycling",
    "Macrophage_LAM",
    "Malignant_MES-hypoxia",
    "Malignant_MES1",
    "Microglia_TREM2_pos",
    "Monocyte_nos",
    "Oligodendrocyte_reactive",
]

sub4_2 = [
    "cDC_CD207_pos",
    "cDC1",
    "cDC2",
    "Perivascular_fibroblast_type-I",
    "Perivascular_fibroblast_cycling",
    "Macrophage_C1Q_pos",
    "Macrophage_IFN_response",
    "Mast_cell",
    "Microglia_intermediate_reactive",
    "NK_KLRC1_pos",
    "NK_nos",
    "pDC",
    "Pericyte_fibro-like",
    "Plasma_cell",
    "Plasmablast",
    "T_cell_CD4",
    "T_cell_CD8",
    "T_cell_IFN_response",
    "T_NK_cycling",
    "T_reg",
]

sub4_3 = [
    "cDC2_cycling",
    "Macrophage_PVM",
]

sub4_4 = [
    "B_cell",
    "Endothelial_arterial",
    "Endothelial_tip",
    "Endothelial_cycling",
    "Endothelial_veinous",
    "Perivascular_fibroblast_type-III",
    "Neutrophil",
    "Other_LowCounts",
    "Pericyte",
    "Pericyte_cycling",
    "VSMC",
    "T_cell_nos",
]

sub_assign_l4 = pd.concat(
    [
        pd.Series(1, index=sub4_1),
        pd.Series(2, index=sub4_2),
        pd.Series(3, index=sub4_3),
        pd.Series(4, index=sub4_4),
    ]
)
sub_assign_l4.name = "submodule"
sub_assign_l4 = sub_assign_l4[~sub_assign_l4.index.duplicated(keep="first")]

print("Module 4 submodule membership:")
print(sub_assign_l4.value_counts().sort_index())

# ------------------------------------------------------------------
# 2) Build per-patient submodule scores from df_l4
#    (df_l4: patients x level4 cell types, proportions)
# ------------------------------------------------------------------

try:
    df_l4
except NameError:
    raise NameError("df_l4 is not defined. Run the cell that builds df_l4 from level 4 proportions first.")

present_cells = [c for c in sub_assign_l4.index if c in df_l4.columns]
missing_cells = sorted(set(sub_assign_l4.index) - set(present_cells))
if missing_cells:
    print("Warning: these submodule cell types are not present in df_l4 and will be ignored:")
    print(", ".join(missing_cells))

sub_assign_present = sub_assign_l4.loc[present_cells]

mod_scores_l4_sub = pd.DataFrame(index=df_l4.index)
for sub in sorted(sub_assign_present.unique()):
    cols = sub_assign_present.index[sub_assign_present == sub]
    colname = f"ModuleL4_4_sub{sub}"
    mod_scores_l4_sub[colname] = df_l4[cols].sum(axis=1)

print("\nModule 4 submodule score means:")
print(mod_scores_l4_sub.mean().sort_values())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# ------------------------------------------------------------------
# KM plot for Module 4 submodule 2 (immune-enriched)
# ------------------------------------------------------------------

MODULE_COL = "ModuleL4_4_sub2"   # which submodule to use
SCHEME = "80_20"                 # "median", "60_40", "70_30", "80_20", "90_10", "extreme_quartiles"
min_group_n = 15                 # do not plot if a group is smaller than this

# module scores per patient (index should be sample_root)
scores = mod_scores_l4_sub[MODULE_COL].copy()

# merge with clinical survival info
df_km = (
    clin_surv[["sample_root", "os_years", "os_event"]]
    .merge(scores.rename("module_score"),
           left_on="sample_root",
           right_index=True,
           how="inner")
    .dropna(subset=["os_years", "os_event", "module_score"])
)

vals = df_km["module_score"].astype(float)

# define split schemes
if SCHEME == "median":
    thr = vals.median()
    mask_high = vals >= thr
    mask_low  = vals < thr

elif SCHEME == "60_40":
    q40, q60 = np.quantile(vals, [0.4, 0.6])
    mask_high = vals >= q60
    mask_low  = vals <= q40

elif SCHEME == "70_30":
    q30, q70 = np.quantile(vals, [0.3, 0.7])
    mask_high = vals >= q70
    mask_low  = vals <= q30

elif SCHEME == "80_20":
    q20, q80 = np.quantile(vals, [0.2, 0.8])
    mask_high = vals >= q80
    mask_low  = vals <= q20

elif SCHEME == "90_10":
    q10, q90 = np.quantile(vals, [0.1, 0.9])
    mask_high = vals >= q90
    mask_low  = vals <= q10

elif SCHEME == "extreme_quartiles":
    q25, q75 = np.quantile(vals, [0.25, 0.75])
    mask_high = vals >= q75
    mask_low  = vals <= q25

else:
    raise ValueError(f"Unknown SCHEME {SCHEME}")

gH = df_km[mask_high]
gL = df_km[mask_low]

print(f"{MODULE_COL} {SCHEME} split: High n={len(gH)}, Low n={len(gL)}")
print(f"median score High={gH['module_score'].median():.4g}, Low={gL['module_score'].median():.4g}")

if len(gH) < min_group_n or len(gL) < min_group_n:
    raise ValueError(f"Too few patients for KM (High={len(gH)}, Low={len(gL)}). Increase groups or lower min_group_n.")

# KM fit and log-rank
kmfH = KaplanMeierFitter(label=f"{MODULE_COL} High")
kmfL = KaplanMeierFitter(label=f"{MODULE_COL} Low")

time_H = gH["os_years"].astype(float)
event_H = gH["os_event"].astype(int)
time_L = gL["os_years"].astype(float)
event_L = gL["os_event"].astype(int)

kmfH.fit(time_H, event_observed=event_H)
kmfL.fit(time_L, event_observed=event_L)

lr = logrank_test(time_H, time_L, event_observed_A=event_H, event_observed_B=event_L)
p = float(lr.p_value)

plt.figure(figsize=(6.0, 5.0))
kmfH.plot(ci_show=True)
kmfL.plot(ci_show=True)

plt.title(f"OS KM: {MODULE_COL} ({SCHEME} split)\nlog-rank p={p:.3g}")
plt.xlabel("Overall survival (years)")
plt.ylabel("Survival probability")
plt.grid(False)
plt.legend(frameon=False, loc="best")
plt.tight_layout()
plt.show()

print(f"Log-rank p-value: {p:.6g}")

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms.community import louvain_communities

# ---------------------------------------------------------------------
# 1. Inputs you already have
# df_l4: patients x celltype_level4_curated proportions
# mod_assign_l4: Series, index = level4 cell types, values = 1..8 (module ID)
# ---------------------------------------------------------------------

# restrict to module 4 cell types
mod4_cells = mod_assign_l4[mod_assign_l4 == 4].index
df_mod4 = df_l4[mod4_cells].copy()

# 2. correlation matrix within module 4
corr_mod4 = df_mod4.corr().astype(float)
corr_mod4 = corr_mod4.fillna(0.0)

# 3. build graph: keep only positive correlations above a small threshold
#    (threshold is just to drop pure noise; you can inspect sensitivity later)
thr = 0.3
A = corr_mod4.clip(lower=0.0)

G = nx.Graph()
for ct in A.index:
    G.add_node(ct)

for i, ct_i in enumerate(A.index):
    for j, ct_j in enumerate(A.columns[i+1:], start=i+1):
        w = float(A.iat[i, j])
        if w >= thr:
            G.add_edge(ct_i, ct_j, weight=w)

print(f"Module 4: {G.number_of_nodes()} cell types, {G.number_of_edges()} edges (r >= {thr})")

# 4. Louvain community detection on this graph
comms = louvain_communities(G, weight="weight", resolution=1.0, seed=0)

# turn communities into a Series: each cell type gets a community label 1..K
comm_labels = {}
for k, comm in enumerate(comms, start=1):
    for ct in comm:
        comm_labels[ct] = k

sub_assign_l4_graph = pd.Series(comm_labels, name="module4_community").sort_values()

print("Module 4 communities from Louvain:")
for k in sorted(sub_assign_l4_graph.unique()):
    members = sub_assign_l4_graph[sub_assign_l4_graph == k].index.tolist()
    print(f"  Community {k} (n={len(members)}): {', '.join(members)}")

# optional: save to CSV so we can freeze this definition
sub_assign_l4_graph.to_csv(f"{LOAD_DIR}/module4_level4_communities_louvain.csv")

In [ ]:
immune_comm_id = 3  # for example, after you inspect the printout
immune_cells = sub_assign_l4_graph[sub_assign_l4_graph == immune_comm_id].index

module_immune = df_l4[immune_cells].sum(axis=1)
module_immune.name = "ModuleL4_immune_graph"

module_immune.to_csv(f"{LOAD_DIR}/moduleL4_immune_graph_scores.csv")
print("Immune community members:", list(immune_cells))
print(module_immune.describe())